## **create an enhanced table which contains summary feedback and feedback rating. These two colums are generate by AI** ##

In [1]:
bronze_catalog    = "bronze_supplier"
silver_catalog    = "silver_supplier"
bronze_schema     = "supplier_schema"
silver_schema     = "supplier_schema"
silver_supplier_basic_Cont_csv = "silver_supplier_basic_Cont_csv"
silver_supplier_feedback_AI_csv = "silver_supplier_feedback_AI_csv"


In [2]:
%sql
drop table if exists silver_supplier.supplier_schema.silver_supplier_feedback_AI_csv

OK

## Create the AI enhanced table if it is not already made ##

In [3]:
cont_df = spark.read.table(f"{silver_catalog}.{silver_schema}.{silver_supplier_basic_Cont_csv}")
cont_df = spark.table (f"{silver_catalog}.{silver_schema}.{silver_supplier_basic_Cont_csv}")
spark.sql("create table if not exists silver_supplier.supplier_schema.silver_supplier_feedback_AI_csv (FEEDBACK_silver  string,  supplier_name_silver  STRING, rating_1_silver  int, rating_2_silver STRING, REViEWTEXT_1 STRING, Summary_silver STRING, rating STRING )")

DataFrame[]

In [4]:
%sql
select * from silver_supplier.supplier_schema.silver_supplier_feedback_AI_csv

In [5]:
spark.sql("select * from silver_supplier.supplier_schema.silver_supplier_feedback_AI_csv  ").show(10, False)


+---------------+--------------------+---------------+---------------+------------+--------------+------+
|FEEDBACK_silver|supplier_name_silver|rating_1_silver|rating_2_silver|REViEWTEXT_1|Summary_silver|rating|
+---------------+--------------------+---------------+---------------+------------+--------------+------+
+---------------+--------------------+---------------+---------------+------------+--------------+------+



## Check the first AI prompt ##

In [6]:
%sql
-- delete from silver_supplier.supplier_schema.silver_supplier_basic_cont_csv where supplier_name_silver iS NULL;

SELECT *, 
       query_model("cohere.command-r-08-2024", 
                   CONCAT("summarize in two sentence: ", REVIEWTEXT_1)) AS output 
FROM bronze_supplier.supplier_schema.bronze_supplier_feedback_csv

## Check the second prompt ##

In [7]:
%sql
SELECT *,        query_model("cohere.command-r-08-2024", 
                   CONCAT("summarize in two sentence: ", REVIEWTEXT_1)) AS output, query_model("cohere.command-r-08-2024", 
                   CONCAT("rate the sentimate from 1-5, 5 is the best, just give me the number: ", output)) as rating
FROM bronze_supplier.supplier_schema.bronze_supplier_feedback_csv

## Now insert it into a new table ##

In [8]:
%sql

insert into silver_supplier.supplier_schema.silver_supplier_feedback_ai_csv (FEEDBACK_silver, supplier_name_silver, rating_1_silver, rating_2_silver, REViEWTEXT_1, Summary_silver, rating) 

SELECT *,        query_model("cohere.command-r-08-2024", 
                   CONCAT("summarize in two sentence: ", REVIEWTEXT_1)) AS output, query_model("cohere.command-r-08-2024", 
                   CONCAT("rate the sentimate from 1-5, 5 is the best, just give me the number: ", output)) as rating
FROM bronze_supplier.supplier_schema.bronze_supplier_feedback_csv

OK

In [9]:
%sql
select supplier_name_silver , Summary_silver, rating  from  silver_supplier.supplier_schema.silver_supplier_feedback_ai_csv